<a href="https://colab.research.google.com/github/juliawol/WB_Sufficiency/blob/main/Notebooks/WB_QA_sufficiency__fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import pandas as pd

# Load and clean the dataset
data = pd.read_csv('/content/qa_dataset_labeled.csv')

# Drop unnecessary columns
data = data.drop(columns=["Question_clean", "Description_clean", "Answer_clean", "True_Class_Prob"], errors='ignore')

# Ensure required columns are present
assert {"Question", "Description", "True_Class"}.issubset(data.columns), "Required columns are missing!"

# Rename 'True_Class' to 'label' for Hugging Face compatibility
data = data.rename(columns={"True_Class": "label"})



In [11]:
# Prepare tokenizer
tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruBERT-large")

def tokenize(batch):
    return tokenizer(batch["Question"], batch["Description"], padding=True, truncation=True, max_length=512)

# Split the data into training and test sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Tokenize the data
train_encodings = tokenizer(list(train_data["Question"]), list(train_data["Description"]), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(list(test_data["Question"]), list(test_data["Description"]), truncation=True, padding=True, max_length=512)

# Convert data into a Dataset object for Hugging Face's Trainer
import torch
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, list(train_data["label"]))
test_dataset = CustomDataset(test_encodings, list(test_data["label"]))

# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained("ai-forever/ruBERT-large", num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Set up the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()

# Save the fine-tuned model and tokenizer
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Model and tokenizer saved!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.475600,0.522228
2,0.326400,0.463025
3,0.254600,0.477661


Model and tokenizer saved!
